# Phase 2 — Nettoyer, filtrer et chiffrer l'entonnoir (J1 après-midi)

In [19]:
import sys
import pandas as pd
import numpy as np
from datetime import datetime

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
data_csv = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)

from validation_structure import normaliser, valider

Intégrez le module fourni. Il renvoie pour chaque numéro un verdict et un motif. Ces motifs ne sont pas tous de même nature : lisez le code pour comprendre ce que chacun signifie exactement, puis décidez du sort de chaque famille. Certaines décisions feront bouger vos chiffres finaux de plusieurs points : sachez lesquelles, et sachez les défendre.

Décidez ensuite de ce qu'est un doublon dans ce référentiel, et comptez-les. Votre définition doit pouvoir s'écrire en une phrase.

Préparez enfin la journée de demain. Appelez le service à la main sur une vingtaine de numéros de pays différents et mesurez son temps de réponse. Décrivez cette distribution comme vous décririez n'importe quelle variable : centre, dispersion, valeurs extrêmes, différences entre pays. Puis faites le calcul : combien de temps prendrait la vérification de tout le référentiel si l'on appelait le service une fois par ligne ? Combien d'appels votre nettoyage vous a-t-il fait économiser ?

Résultat attendu en fin de J1 : un tableau qui va de 10 000 lignes au nombre exact de numéros qui méritent un appel, étape par étape, avec la répartition par motif de rejet et une estimation chiffrée de la durée d'une campagne complète.

#### Le modèle de l'API

Définitions 

VIES : VAT Information Exchange System.

Moteur de recherche gratuit de la Commission européenne qui interroge directement les bases de données fiscales nationales pour : 
- Facturer sans TVA (hors taxes), 
- Vérifier un fournisseur, 
- Se protéger en cas de contrôle fiscal

le formulaire : 
https://ec.europa.eu/taxation_customs/vies/#/vat-validation

exemple de requête :
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/27552032534

Le formulaire ne demande que deux champs : l'état membre et le numéro de TVA. Nous retiendrons ces deux critères parmis les données.

#### Je récapitule les motifs avec leurs parts

In [ ]:
data_csv["tva_normalise"] = data_csv["numero_tva"].apply(normaliser)
resultats = data_csv["tva_normalise"].apply(valider)
data_csv["est_valide"] = [res[0] for res in resultats]
data_csv["motif"] = [res[1] for res in resultats]
display(data_csv["motif"].value_counts())

motif
ok                  6083
cle_invalide        2605
pays_non_couvert    1090
vide                 174
trop_court            48
Name: count, dtype: int64

`cle_invalide` sera refusé par l'API, comme `vide` et `trop_court`. 

Je regarde `pays_non_couvert` en détail pour prendre une décision :

In [21]:
display(data_csv.loc[data_csv["motif"] == "pays_non_couvert", ["numero_tva", "pays_declare", "motif"]].head(20))

,numero_tva,pays_declare,motif
16,GB0749640348,GB,pays_non_couvert
34,ZZ23140153,ZZ,pays_non_couvert
68,GB17652291,GB,pays_non_couvert
79,UK227979060,UK,pays_non_couvert
119,58560322811,FR,pays_non_couvert
134,396904956B68,NL,pays_non_couvert
145,UK190500864B18,UK,pays_non_couvert
151,ZZ06520766,ZZ,pays_non_couvert
162,38486825359,IT,pays_non_couvert
180,33139885439,IT,pays_non_couvert


Il y a des valeurs qui ve renvoient pas à des pays (XX, ZZ, QQ).

GB et UK ne font pas partie de la liste CONTROLES de `validation_structure.py` : ils ne font pas partie de l'Union Européenne. Un message adressant les UK VATs apparait sur la page d'acceuil de la page de requête : le service pour les valider a cessé d'exister. De même, un status spécial `XI` est attribué à l'Irlande du Nord, membre du Royaume Uni, qui est hors UE.

Certaines valeurs de la colonne `pays_declare` renvoient à des pays qui font pourtant partie de la liste `CONTROLES` : c'est parce que le calcul d'attribution de  l'étiquette `pays_non_couvert` sur fait sur `data_csv["numero_tva"]` : c'est le numéro TVA qui ne renvoie pas à un pays couvert dans sa forme, et on le constate en regardant les valeurs de `numero_tva` ci dessus, avec les préfixes manquants.

Alors, je décide d'écarter les valeurs `pays_non_couvert` de mon corpus final. Il faut maintenant tester les 6083 valeurs retenues étiquettées du motif `ok` pour vérifier s'il y a des doublons.

#### Qu'est-ce qu'un doublon ? 

Un doublon est une valeur qui apparait deux fois dans un corpus (dataframe).

Cela fausse les résultats car cela donne une pondération plus importante à un élément unique. 

Il faut donc retirer les doublons.

In [22]:
print(f"Il y a {data_csv.duplicated().sum()} lignes strictement identiques")

print(f"Il y a {data_csv["tva_normalise"][data_csv["motif"] == "ok"].duplicated().sum()} doublons dans la colonne 'numero_tva' où la valeur 'motif' correspond à 'ok'")
# Je prends la colone tva_normalise (et pas numero_tva) et je ne veux garder que les lignes ou le motif est "ok", puis je compte le total (sum) de duplicats (duplicated)

#Je créé un duplicata de la table
csv_ok_avec_doublons = data_csv.loc[data_csv["motif"] == "ok"].copy()
print(f"Nombre de lignes dans csv_ok_avec_doublons : {len(csv_ok_avec_doublons)}")

# Je retire les doublons 
csv_ok_sans_doublon = csv_ok_avec_doublons.drop_duplicates(subset=["tva_normalise"], keep="first")
print(f"nombre de lignes dans csv_ok_sans_doublon : {len(csv_ok_sans_doublon)}")

# le compte devrait être égal à la somme de "ok" moins le nombre de doublons (6083 - 281, soit 5802)


Il y a 0 lignes strictement identiques
Il y a 281 doublons dans la colonne 'numero_tva' où la valeur 'motif' correspond à 'ok'
Nombre de lignes dans csv_ok_avec_doublons : 6083
nombre de lignes dans csv_ok_sans_doublon : 5802


#### Je prépare la journée de demain 

In [23]:
# créer une variable data_csv_nettoye avec une colonne tva_normalise sans préfix

csv_ok_sans_doublon["tva_sans_pays"] = csv_ok_sans_doublon["tva_normalise"].str[2:]
display(csv_ok_sans_doublon[["tva_normalise", "pays_declare", "tva_sans_pays"]].head())

,tva_normalise,pays_declare,tva_sans_pays
1,PT787327239,PT,787327239
2,LU71043234,LU,71043234
3,SE751298146001,SE,751298146001
5,DK61530843,DK,61530843
7,SE314125231601,SE,314125231601


In [24]:
# dernière vérification que le préfixe concorde avec tva_normalise
prefixe_tva = csv_ok_sans_doublon["tva_normalise"].str[:2]
incoherences = csv_ok_sans_doublon.loc[prefixe_tva != csv_ok_sans_doublon["pays_declare"]]
print(f"Nombre d'incohérences trouvées : {len(incoherences)}")
display(incoherences[["numero_tva", "tva_normalise", "pays_declare"]])

Nombre d'incohérences trouvées : 0


,numero_tva,tva_normalise,pays_declare


- Appelez le service à la main sur une vingtaine de numéros de pays différents et mesurez son temps de réponse. 

Je fais un appel par pays figurant dans CONTROLES : 

| pays | tva_sans_pays | temps de réponse |
| --- | --- | --- |
| FR | 63432337650 | --- |
| BE | 1997422119 | --- |
| NL | 767957817B57 | --- |
| LU | 71043234 | --- |
| PT | 787327239 | --- |
| DK | 61530843 | --- |
| FI | 84564443 | --- |
| PL | 7963612947 | --- |
| IT | 97133603:raci924 | --- |
| SE | 751298146001 | --- |

- Décrivez cette distribution comme vous décririez n'importe quelle variable : centre, dispersion, valeurs extrêmes, différences entre pays. 

- Puis faites le calcul : combien de temps prendrait la vérification de tout le référentiel si l'on appelait le service une fois par ligne ? 

- Combien d'appels votre nettoyage vous a-t-il fait économiser ?

### Trop de numéros non actifs qui renvoient `INVALID`, je passe à la suite du brief

In [25]:
display(csv_ok_sans_doublon.head(5))

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_normalise,est_valide,motif,tva_sans_pays
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,True,ok,787327239
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,True,ok,71043234
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,True,ok,751298146001
5,6,Kappa Logistics AB,DK,DK61530843,2025-01-14,import_fournisseur,DK61530843,True,ok,61530843
7,8,Vertex Industries SARL,SE,SE.314125231601,2025-03-31,reprise_erp,SE314125231601,True,ok,314125231601


### Conclusion 


Résultat attendu en fin de J1 : un tableau qui va de 10 000 lignes au nombre exact de numéros qui méritent un appel, étape par étape, avec la répartition par motif de rejet et une estimation chiffrée de la durée d'une campagne complète.

In [26]:
# J'exporte le nouveau CSV et je vérifie les valeurs nulles (= un point de verif random)

csv_ok_sans_doublon.to_csv("tva_propre.csv", index=False)

verif = pd.read_csv("tva_propre.csv")

display(verif[["pays_declare", "tva_normalise"]].isna().sum())

pays_declare     0
tva_normalise    0
dtype: int64

Les motifs de rejets sont documentés et chiffrés plus haut, pour rappel : ok : 6083, cle_invalide : 2605, pays_non_couvert : 1090, vide : 174, trop_court : 48. 

Parmi les 6083 "ok", on a retiré 281 doublons. Il reste 5802 lignes qui méritent un appel. 

Les colones "tva_sans_pays" et "pays_declare" sont préparées en amont pour êtres incluses dans un f string.

In [27]:
### J4 : Ajout d'une sortie de rapport de conciliation

sortie = "rapport_de_conciliation.csv"

masque_doublon = (data_csv["motif"] == "ok") & data_csv.duplicated(
    subset=["tva_normalise"], keep="first"
)
data_csv.loc[masque_doublon, "motif"] = "doublon"

data_csv["statut_nettoyage"] = np.where(data_csv["motif"] == "ok", "Conservée", "Écartée")
data_csv["dernier_controle"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

data_csv.to_csv(sortie, sep=";", index=False, encoding="utf-8-sig")

print("=== BILAN DU NETTOYAGE ===")
print(f"Total des lignes analysées : {len(data_csv)}\n")

print("--- Répartition globale ---")
print(data_csv["statut_nettoyage"].value_counts().to_string())

print("\n--- Détail par motif ---")
print(data_csv["motif"].value_counts().to_string())


=== BILAN DU NETTOYAGE ===
Total des lignes analysées : 10000

--- Répartition globale ---
statut_nettoyage
Conservée    5802
Écartée      4198

--- Détail par motif ---
motif
ok                  5802
cle_invalide        2605
pays_non_couvert    1090
doublon              281
vide                 174
trop_court            48
